# SIADS 521 — Assignment 3
## Divya Drishti — How Widespread Are Food Allergens in U.S. Medications?
*Phase 1 of Sanjeevini Aushad · interactive dashboard*



## 1. Introduction

**The problem.** More than **33 million** Americans live with food allergies — about **1 in 10 adults** and **1 in 13 children** (FARE, 2026). But food allergens do not live only in food: they are also present inside **medications**, hidden among the "inactive" ingredients (excipients) — for example, **lactose** (a milk derivative) is one of the most common tablet fillers.

A patient with a documented food allergy can therefore be prescribed a medication whose excipients contain their allergen, because prescribing and dispensing workflows rarely cross-check inactive ingredients against an allergy profile. Finding a compatible medication can take **more than one try** - the patient effectively becomes the system integrator (see the journey below).


<img src="FAREConsumerJourney.png" width="520">

*The food-allergy patient journey to a compatible medication, often more than one try. Source: FARE Consumer Journey Infographic.*

### Why I built this - my story

In **March 2025**, I was hit by abdominal pain so severe I could barely move. I had reacted to dairy before, but this was relentless, and cutting out foods did nothing. I tried every diet - liquids, elimination, vegan - and saw gastroenterologists, an allergist, my PCP, and integrative specialists. Endoscopy and MRI confirmed internal inflammation, but **no one could tell me why.** The medications they added only made me worse; the only thing offered was "symptom management."

The breakthrough came from a pharmacist's offhand line: *"It could be your meds."* That night I went through every prescription, checking **active *and* inactive** ingredients, and found **lactose in three new medications and one I had taken for years.** I had tolerated small amounts before; the **cumulative** dose overwhelmed my system.

What broke me was not only the pain, it was the **systemic failure.** My lactose reaction was documented across **MyChart and pharmacy records**, yet no doctor or pharmacist flagged it. Everyone passed the ball: *"not my expertise," "outside my job scope," "call the manufacturer."* I spent **2 to 3 weeks on the phone**, calls running over three hours, doing the detective work myself - finding lactose-free formulations (swapping **AbbVie's levothyroxine, which contains lactose, for Pfizer's Levoxyl, which does not**), then coordinating prescriptions, insurance, and providers to get them dispensed.

It cost roughly **$50,000** - insurance covered most, but it was still my money, and far more of my time and health. **I became the system integrator for my own safety.**

> **The gap:** no system cross-checks a patient's known allergy against a medication's *inactive* ingredients, so the patient is left to debug it alone. **Divya Drishti is my first step toward closing that gap: measuring how widespread the problem actually is.**


### This isn't just me - the scale of the problem

FARE counts the **top-9 food allergens** (~**33 million** Americans). But this project also tracks **alpha-gal**, which FARE does **not** cover - and its numbers reveal a hidden scale:

**Alpha-gal syndrome (the 10th U.S. food allergen, outside FARE's top-9):**
- **Up to 450,000 Americans** have alpha-gal syndrome (CDC).
- **60-75%** experience **anaphylaxis**; new suspected cases grow by **more than 15,000 per year**.
- 🔴 **Over 10 confirmed deaths from alpha-gal reactions to *medical products alone*** - precisely the gap this project targets.
- A 2023 CDC survey found **42% of providers were unaware of AGS** (and 35% were not confident managing it) - the same systemic blind spot in my own story.
- In the hardest-hit regions, **up to 3%+ of the population** is affected.
- *Source: Alpha-gal Information / CDC - alphagaldocs.org/facts-and-figures*

**And the problem is documented across the field:**
- **ADINA Act** - proposed U.S. legislation requiring allergen labeling on medications: adinaact.com
- **Beyond Celiac** - gluten hidden in medication excipients: https://www.beyondceliac.org/living-with-celiac-disease/gluten-in-medication/
- **EMJ Reviews** - hidden allergens found in common medications: https://www.emjreviews.com/en-us/amj/allergy-immunology/news/hidden-allergens-found-in-common-medications/
- **Reker et al., *Science Translational Medicine* (2019)** - ~92.8% of oral solid medications contain at least one potential allergen.https://pmc.ncbi.nlm.nih.gov/articles/PMC7122736/ 


### The current user journey (what a patient faces today)
Before a tool like this exists, finding a compatible medication falls entirely on the patient: get the prescription, check active **and** inactive ingredients, identify allergen concerns, verify with the pharmacist / prescriber / manufacturer, explore alternative brands, weigh cost and coverage, re-check - and often loop back to the start.

<img src="Medication Ingredient-Safety Journey (1).png" width="860">

*The current medication ingredient-safety journey - today, the patient becomes the system integrator.*

**The question this dashboard explores:** *How widespread are major food allergens - including alpha-gal (mammalian-derived) ingredients - across FDA-approved medications, and does the prevalence depend on how a drug is taken (its route of administration)?*

## 2. Library choice and fit  *(rubric: 15 pts)*

**Library used:** [**Plotly**](https://plotly.com/python/) (`plotly.express`).

Facts to state (fill these in / verify):
- **Creator:** Plotly Technologies Inc.  ·  **License:** MIT (open source).
- **Install:** `pip install plotly`.
- **Declarative or procedural?** `plotly.express` is **declarative** (describe the chart from a dataframe); `plotly.graph_objects` is the lower-level **procedural** API.
- **Jupyter integration:** renders interactive charts inline in the notebook; exports to a **standalone HTML** file (important for the "publicly runnable" requirement).
- **Main limitations:** can get heavy on very large datasets; fully linked multi-chart dashboards usually need **Dash** (a running server) rather than a static notebook.

**✍️ YOUR WORDS — why Plotly fits *this* data, and one alternative you rejected:**
> *(e.g., "I chose Plotly because my data is categorical and I need interactivity + an exportable HTML dashboard. I considered **Seaborn** but rejected it because its charts are static — they can't give me the hover detail or the linked filtering the assignment requires.")*  ← rewrite in your own voice.


## 3. Data

**Source:** openFDA **Drug Label** dataset — public, U.S.-government (public domain).
Download page: https://open.fda.gov/data/downloads/

**The full dataset is 14 JSON files (~250,000 drug labels).** For this initial analysis I use **file 1 of 14** — `drug-label-0001-of-0014.json` (**20,000 records**). One shard is enough to measure the *scale* of the problem; the full 14-file run is future work.
Direct file: `https://download.open.fda.gov/drug/label/drug-label-0001-of-0014.json.zip`

### Why I kept the columns I kept
- **Ingredients live in *more than one* field.** Active and inactive ingredients are spread across several fields (`inactive_ingredient`, `active_ingredient`, `substance_name`, `spl_product_data_elements`, `description`). So I scan **all approved ingredient fields**, not just one — missing a field could miss an allergen.
- **What the medication treats.** I keep `indications_and_usage` (what issue the drug fixes) so I can later analyze allergen burden **by condition** — my focus conditions are **diabetes, hypertension, and hypothyroidism**.
- **Allergen *source* traceability.** For every allergen I flag, I record **where** I found it (which field + the exact matched text) in an `*_explanation` column — so the evidence trail is never broken. This makes the result **auditable**, which a clinical-decision-support tool must be.
- **Route of administration.** I clean and group the `route` field (oral / injectable / topical / inhaled / other), because *how* a drug is taken strongly affects which excipients it carries — and it enables the oral-vs-injectable comparison below.

### Cleaning steps (done by the `01` / `02` pipeline)
1. **Allergen detection** — a deterministic, rule-based scan of the approved ingredient fields, tagging each of 7 allergens as **Detected / Review / Not Detected / Unknown**, with exclusion rules (e.g., "coconut milk" does not count as milk).
2. **Route cleaning** — the raw `route` is multi-valued (`DENTAL | ORAL | TOPICAL`); it is split, normalized, and turned into Yes/No group flags.


### Cleaning step, shown in code: the multi-valued `route` field

The raw `route` field is **multi-valued** — one medication can list several routes glued together, e.g. `DENTAL | ORAL | TOPICAL`. That cannot be filtered directly (`"DENTAL | ORAL | TOPICAL"` is not equal to `"ORAL"`). So we **split** it on the `|`, **normalize** each route (strip, uppercase, de-duplicate), and turn it into **Yes/No group flags** — a drug that is `ORAL | INTRAVENOUS` becomes `is_oral = True` *and* `is_injectable = True`. (This is a readable version of `clean_route()` in `01_DatasetCreation_4Status.py`.)

In [43]:
import re
import pandas as pd

ORAL = {"ORAL", "SUBLINGUAL", "BUCCAL"}            # all administered in the mouth
TOPICAL = {"TOPICAL", "CUTANEOUS", "TRANSDERMAL", "PERCUTANEOUS"}
INHALED = {"RESPIRATORY (INHALATION)", "NASAL"}

def route_flags(raw):
    routes = sorted({p.strip().upper() for p in re.split(r"\s*\|\s*", str(raw))
                     if p.strip() and p.strip().lower() != "nan"})
    injectable = lambda r: r.startswith("INTRA") or r in {
        "INTRAVENOUS", "INTRAMUSCULAR", "SUBCUTANEOUS", "EPIDURAL", "PARENTERAL",
        "INFILTRATION", "PERINEURAL", "SOFT TISSUE", "ENDOTRACHEAL",
        "EXTRACORPOREAL", "SUBCONJUNCTIVAL"}
    return {
        "route_clean": " | ".join(routes),
        "is_oral": any(r in ORAL for r in routes),
        "is_injectable": any(injectable(r) for r in routes),
        "is_topical": any(r in TOPICAL for r in routes),
        "is_inhaled": any(r in INHALED for r in routes),
    }

# before -> after, on real route strings from the dataset
examples = ["DENTAL | ORAL | TOPICAL", "ORAL | INTRAVENOUS", "INTRAVENOUS", ""]
pd.DataFrame([{"raw route": e or "(blank)", **route_flags(e)} for e in examples])

,raw route,route_clean,is_oral,is_injectable,is_topical,is_inhaled
0,DENTAL | ORAL | TOPICAL,DENTAL | ORAL | TOPICAL,True,False,True,False
1,ORAL | INTRAVENOUS,INTRAVENOUS | ORAL,True,True,False,False
2,INTRAVENOUS,INTRAVENOUS,False,True,False,False
3,(blank),,False,False,False,False


> 🔧 **TODO — after the cleaning is finalized**, add code here to:
> - load the first-level dataset we generated into a dataframe,
> - show `df.head()` and the **record count** (shape / size),
> - show a **pivot table** that illustrates the analysis (e.g. allergen status by route),
>
> so the reader can see exactly what the dataset we created looks like.

### 📏 Counting rule (important)
A single medication can list **several routes of administration** - for example `ORAL | INTRAVENOUS | VAGINAL`. For the **route-level** analysis, such a drug is counted **once for each of its routes**: a drug given 4 ways counts as **4** in the route breakdown. So the per-route counts add up to more than the number of distinct medications - that is expected.

**How big is this effect here?** Small - only **114 medications (0.6%)** list 2 or more routes; 80% list none. So the exploded count is **20,165** drug-route rows, barely above the **20,000** distinct medications. We therefore report the headline on **20,000 distinct medications**, and apply the per-route counting only inside the route-level charts (the route-group flags let each drug appear in every route it belongs to).


## 4. Visualization techniques  *(rubric: 20 pts)*

For each chart, state **what it shows**, **why here**, and **when NOT to use it**. Starter notes:

- **Pie (overall status):** shows parts of a single whole (every medication is exactly one of Detected / Review / Not Detected). *When not to use:* when categories overlap or there are many slices.
- **Bar (per allergen):** compares the 7 allergens' prevalence. Right choice here because a drug can have **several** allergens, so the shares overlap — a pie would be misleading.
- **✍️ YOUR WORDS — your new-to-course method** (not taught in 521): e.g., a **network / chord** of allergen *co-occurrence*, or **parallel coordinates**. Explain what it reveals that a bar/pie cannot.


## 5. Building and exploring  *(rubric: 35 pts — the biggest)*

Build up from simple charts to the linked, interactive dashboard.

*(We'll revisit this whole section together — the charts below are a starting point.)*

In [44]:
import pandas as pd
import plotly.express as px

# load the first-level dataset we generated
df = pd.read_csv("output/01_DatasetCreation_4Status_Output.csv", low_memory=False)
for c in ["has_route", "is_oral", "is_injectable", "is_topical", "is_inhaled", "is_other_route"]:
    df[c] = df[c].astype(str).str.lower().isin(["true", "1"])

### Chart 1 — How bad is it? (overall status)

In [45]:
overall = pd.read_csv("output/overall_allergen_summary.csv")
overall = overall[overall["Count"] > 0]           # drop the empty 'Unknown' slice

fig = px.pie(overall, names="Overall_Status", values="Count", hole=0.35,
             title="Of 20,000 FDA medications, how many flag a food allergen?",
             color="Overall_Status",
             color_discrete_map={"Detected": "#e4572e", "Review": "#f4a259", "Not Detected": "#7fb069"})
fig.update_traces(textinfo="label+percent")
fig.show()

### Chart 2 — Which allergen dominates? (per-allergen prevalence)

In [46]:
al = pd.read_csv("output/02_DatasetCreation_AllergenSummary_Output.csv")

fig = px.bar(al.sort_values("Detected_Pct"), x="Detected_Pct", y="Allergen",
             orientation="h", text="Detected_Pct",
             title="Share of medications where each allergen is Detected",
             labels={"Detected_Pct": "% of 20,000 medications", "Allergen": ""})
fig.update_traces(texttemplate="%{text:.1f}%")
fig.show()

### Chart 3 — The finding: does route matter?
*Milk is a tablet filler (lactose), so it should concentrate in **oral** medicines and be near-absent in injectables.*

> ⚠️ **Important limitation (applies to Charts 3-5).** Only about **20% of the 20,000 medications have a route recorded** in openFDA; the other **~80% are blank** because FDA never mapped them to its structured fields. So every route-based chart reflects only the **3,977 medications that *have* a route**, not all 20,000. The pattern is clear within that subset, but it is not representative of the full catalogue - recovering route from unstructured dosage-form text is noted as future work.

In [47]:
groups = {"oral": "is_oral", "injectable": "is_injectable", "topical": "is_topical",
          "inhaled/nasal": "is_inhaled", "other-local": "is_other_route"}
rows = []
for label, col in groups.items():
    sub = df[df[col]]
    n = len(sub)
    d = (sub["milk_status"] == "Detected").sum()
    rows.append({"route": label, "milk_pct": round(d / n * 100, 1) if n else 0, "n_drugs": n})
route_milk = pd.DataFrame(rows).sort_values("milk_pct", ascending=False)

fig = px.bar(route_milk, x="route", y="milk_pct", text="milk_pct",
             title="Milk prevalence by route — oral pills vs. everything else",
             labels={"milk_pct": "% of that route's drugs with milk Detected", "route": ""},
             hover_data=["n_drugs"])
fig.update_traces(texttemplate="%{text}%")
fig.show()
route_milk

,route,milk_pct,n_drugs
0,oral,37.6,2263
2,topical,8.6,1194
4,other-local,5.3,151
1,injectable,2.3,298
3,inhaled/nasal,1.2,82


### Chart 4 — Heatmap: allergen prevalence within each route
*(Note: the heatmap IS taught in 521, so it is a strong 4th chart but not the "new" method.)*

In [48]:
ALLERGENS = {"Milk": "milk_status", "Eggs": "eggs_status", "Wheat": "wheat_status",
             "Sesame": "sesame_status", "Tree Nuts": "tree_nuts_status",
             "Peanuts": "peanuts_status", "Soy": "soy_status", "Alpha-Gal": "alpha_gal_status"}
ROUTES = {"oral": "is_oral", "injectable": "is_injectable", "topical": "is_topical",
          "inhaled/nasal": "is_inhaled", "other-local": "is_other_route"}

mat, ylabels = [], []
for rlabel, rcol in ROUTES.items():
    sub = df[df[rcol]]
    if len(sub) == 0:
        continue
    mat.append([round((sub[acol] == "Detected").mean() * 100, 1) for acol in ALLERGENS.values()])
    ylabels.append(f"{rlabel} (n={len(sub)})")

fig = px.imshow(mat, x=list(ALLERGENS.keys()), y=ylabels, text_auto=True, aspect="auto",
                color_continuous_scale="Reds", labels=dict(color="% Detected"),
                title="Allergen prevalence (%) within each route")
fig.show()

### Chart 5 ⭐ NEW-TO-COURSE METHOD — Sunburst: route then allergen
A **sunburst** is a set of nested rings you can click into: the inner ring is the route, and clicking it drills down to the allergens inside that route. This is **not** a method taught in SIADS 521, so it satisfies the "new method" requirement. It shows what a bar cannot: the *nested* structure (within oral pills, which allergens dominate).

In [49]:
rows = []
for rlabel, rcol in ROUTES.items():
    sub = df[df[rcol]]
    for aname, acol in ALLERGENS.items():
        cnt = int((sub[acol] == "Detected").sum())
        if cnt > 0:
            rows.append({"route": rlabel, "allergen": aname, "count": cnt})
sb = pd.DataFrame(rows)

fig = px.sunburst(sb, path=["route", "allergen"], values="count", color="route",
                  title="Sunburst - food allergens detected, by route then allergen")
fig.update_traces(textinfo="label+value")
fig.show()

### Chart 6 ⭐ NEW METHOD — Waffle: how many allergens does one medication carry?
A **waffle chart** is a 10x10 grid of 100 squares, each ~1% of medications, colored by how many allergens that medication has **Detected**. Not taught in SIADS 521. It answers *"how many pills are clean vs. multi-allergen minefields?"* at a glance. *(Detected only - "Review" is excluded, or Alpha-Gal's ubiquitous ambiguous excipients would swamp the chart.)*

In [50]:
import numpy as np
import plotly.graph_objects as go

status_cols = [c for c in df.columns if c.endswith("_status")]
n_det = sum((df[c] == "Detected").astype(int) for c in status_cols)

def _bucket(n):
    return "0 (none)" if n == 0 else ("1" if n == 1 else ("2" if n == 2 else "3 or more"))

order = ["0 (none)", "1", "2", "3 or more"]
counts = n_det.apply(_bucket).value_counts().reindex(order, fill_value=0)
pct = counts / counts.sum() * 100

# allocate 100 squares (largest-remainder rounding)
base = np.floor(pct).astype(int)
for i in np.argsort(-(pct - base).values)[: int(100 - base.sum())]:
    base.iloc[i] += 1

colors = {"0 (none)": "#7fb069", "1": "#f4a259", "2": "#e4572e", "3 or more": "#9b2226"}
seq = [cat for cat in order for _ in range(int(base[cat]))]

fig = go.Figure(go.Scatter(
    x=[i % 10 for i in range(len(seq))], y=[i // 10 for i in range(len(seq))], mode="markers",
    marker=dict(size=24, symbol="square", color=[colors[c] for c in seq],
                line=dict(width=2, color="white")),
    text=seq, hoverinfo="text"))
for cat in order:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
        marker=dict(size=12, symbol="square", color=colors[cat]),
        name=f"{cat}: {int(counts[cat]):,} meds ({pct[cat]:.1f}%)"))
fig.update_layout(
    title="Waffle - each square = ~1% of medications, by number of allergens Detected (of 8)",
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"),
    plot_bgcolor="white", width=640, height=560, legend=dict(orientation="h", y=-0.08))
fig.show()

### ✍️ YOUR WORDS — exploration narrative *(rubric rewards this heavily)*
What did you *start* asking? What surprised you? What did a **dead end** look like? What did you see **only through interaction**?
> *(e.g., "I expected allergens to be spread evenly, but milk dominated. I then split by route and found oral pills carry ~16× the milk rate of injectables — a pattern I only saw after adding the route filter.")*

### 🔧 TODO — linked interactivity (rubric requirement #4)
Add **one control that updates two or more charts at once** (e.g., a route dropdown that re-filters both the status pie and the allergen bar). At least one finding must be visible *only* through that interaction.


## 6. AI log and troubleshooting  *(rubric: 10 pts)*

AI tools were used and must be disclosed. Include **at least 3 entries**, **≥2 showing something AI got wrong** and how you caught/fixed it.

*You have real material from building this:* an AI tool misremembered a project's architecture; a rule-based tagger mis-labeled an ingredient; a local LLM gave a self-contradicting verdict; another AI over-claimed on a résumé. Use the template:

```
Entry 1: <short title>
- Tool: (Claude / ChatGPT / Copilot / Gemini / none)
- What I asked for / was trying to do:
- What came back / what happened:
- What was wrong or could be better:
- How I noticed:
- What I did instead:
```
**✍️ YOUR WORDS — write at least 3 entries here.**


## 7. Deployment  *(publicly runnable — essential)*

The dashboard must open and work for anyone with the link, nothing installed.

Options (fill in what you actually do):
- **Standalone HTML** (Plotly supports this): `fig.write_html("dashboard.html")` → upload to Canvas / repo. Opens in any browser, no server.
- **Hosted:** Hugging Face Spaces or Streamlit Community Cloud for a fuller linked dashboard.

**✍️ YOUR WORDS — step-by-step deployment a classmate could repeat.**

---
*Repo: `ushajaki-u/Sanjeeva-Aushadh` · Data: openFDA Drug Label (public domain) · Benchmark: Reker et al., Sci Transl Med 2019.*
